# FormReader - TÜBİTAK 3501 Projesi

## Hücre Rehberi
- **Hücre 1-4**: Form işleme (opsiyonel)
- **Hücre 5**: Sentetik veri üretimi (50K) - ~30-40 dk
- **Hücre 6**: Model eğitimi (Deney #003 Bug Fix) - ~7-8 saat
- **Hücre 7**: Sonuç testi (DENEY değişkenini değiştirerek farklı deneyleri test et)

---
**GECE İÇİN**: Sadece Hücre 6'yı çalıştır! (veri zaten mevcut)

**VERSİYONLAMA**: Her deney ayrı klasöre kaydedilir (`deney002/`, `deney003/`, ...)

In [ ]:
# ============================================================
# HÜCRE 1: Kütüphaneleri Yükle
# ============================================================
import sys
import os
import cv2
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('../src')) 

from preprocessing import find_boxes
from utils import parse_xml_labels, match_and_crop
from ocr_engine import OCREngine

def show(img, title="Resim"):
    plt.figure(figsize=(10,10))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

print("[OK] Kütüphaneler yüklendi")

In [ ]:
# ============================================================
# HÜCRE 2: Formu Oku ve Kutuları Bul (OPSİYONEL)
# ============================================================
IMAGE_PATH = "../data/raw/S25C-925103023031.jpg"

processed_image, boxes = find_boxes(IMAGE_PATH)
print(f"Toplam {len(boxes)} adet kutu bulundu.")
show(processed_image, "Bulunan Kutular")

In [ ]:
# ============================================================
# HÜCRE 3: Etiketleri Eşleştir ve Kes (OPSİYONEL)
# ============================================================
XML_PATH = "../data/xml_labels/S25C-925103023031.xml"

xml_labels = parse_xml_labels(XML_PATH)
labeled_data = match_and_crop(processed_image, boxes, xml_labels, output_folder="../data/processed")

for item in labeled_data[:5]:
    print(f"Etiket: {item['label']}")
    show(item['roi_image'], item['label'])

In [ ]:
# ============================================================
# HÜCRE 4: Base Model ile Test (OPSİYONEL)
# ============================================================
ocr = OCREngine()

print("\n--- BASE MODEL OKUMA SONUÇLARI ---")
for item in labeled_data[:10]:
    text = ocr.predict(item['roi_image'])
    print(f"Kutu: {item['label']} -> Okunan: {text}")

In [ ]:
# ============================================================
# HÜCRE 5: SENTETİK VERİ ÜRETİMİ (200.000 ADET)
# ============================================================
# Tahmini süre: ~2 saat
# Disk: ~8-10 GB
# ============================================================

from data_generator import generate_dataset
import pandas as pd

FONT_DIR = "../data/fonts"
OUTPUT_DIR = "../data/synthetic"
COUNT = 200000  # Deney #005 için 200K

print("="*60)
print("SENTETİK VERİ ÜRETİMİ BAŞLIYOR")
print(f"Hedef: {COUNT:,} görüntü")
print("="*60)

csv_path = generate_dataset(FONT_DIR, OUTPUT_DIR, count=COUNT)

# Kontrol
df = pd.read_csv(csv_path)
print(f"\n[TAMAMLANDI] Toplam: {len(df):,} veri")
print(f"\nKategori dağılımı:")
print(df['category'].value_counts())

In [ ]:
# ============================================================
# HÜCRE 6: MODEL EĞİTİMİ (Deney #005)
# ============================================================
# 3 İyileştirme:
#   1. Smart Embedding Init: ş←s, ğ←g, ü←u ağırlık kopyalama
#   2. Unicode NFC Normalization: Türkçe karakter tutarlılığı
#   3. Encoder Freeze: ViT donduruldu, sadece decoder öğreniyor
# GPU: RTX 4070 Super 12GB
# ============================================================

import importlib
import trainer
importlib.reload(trainer)
from trainer import train_trocr

DENEY = "deney005"
EPOCHS = 5

print("="*60)
print(f"DENEY #005: TrOCR + Türkçe Tokenizer")
print("(Smart Init + NFC + Encoder Freeze)")
print("="*60)
print(f"Epochs: {EPOCHS}")
print(f"Kayıt: models/trocr-turkish-handwritten/{DENEY}/")
print("="*60)

train_trocr(
    dataset_dir="../data/synthetic", 
    csv_path="../data/synthetic/metadata.csv", 
    output_model_dir=f"../models/trocr-turkish-handwritten/{DENEY}",
    epochs=EPOCHS,
    batch_size=8,
    learning_rate=2e-5
)

print("\n" + "="*60)
print("EĞİTİM TAMAMLANDI!")
print(f"Model kaydedildi: models/trocr-turkish-handwritten/{DENEY}/")
print("Sonuçları görmek için Hücre 7'yi çalıştır.")
print("="*60)

In [ ]:
# ============================================================
# HÜCRE 7: SONUÇ TESTİ
# ============================================================

import importlib
import ocr_engine
importlib.reload(ocr_engine)
from ocr_engine import OCREngine
import pandas as pd
from PIL import Image
import os

# ===== DENEY SEÇİMİ =====
DENEY = "deney005"
# =========================

model_path = f"../models/trocr-turkish-handwritten/{DENEY}"

if not os.path.exists(model_path):
    model_path = "../models/trocr-turkish-handwritten"
    print(f"[Bilgi] Deney klasörü bulunamadı, eski format deneniyor: {model_path}")

print(f"Test edilen deney: {DENEY}")
print(f"Model yolu: {model_path}")
print()

print("Fine-tuned model yükleniyor...")
ocr = OCREngine(model_path=model_path)

print("\nModel Bilgisi:")
for k, v in ocr.get_model_info().items():
    print(f"  {k}: {v}")

print("\n" + "="*60)
print(f"SENTETİK VERİ TESTİ ({DENEY})")
print("="*60)

df = pd.read_csv("../data/synthetic/metadata.csv")

categories = ['number', 'turkish_text', 'date', 'mixed', 'code']

correct = 0
total = 0

for cat in categories:
    print(f"\n--- {cat.upper()} ---")
    samples = df[df['category'] == cat].sample(5, random_state=42)
    
    for _, row in samples.iterrows():
        img_path = os.path.join("../data/synthetic", row['file_name'])
        expected = str(row['text'])
        predicted = ocr.predict(img_path)
        
        match = "✓" if predicted.strip() == expected.strip() else "✗"
        if match == "✓":
            correct += 1
        total += 1
        
        print(f"{match} Beklenen: {expected:<25} | Tahmin: {predicted}")

print("\n" + "="*60)
print(f"DENEY: {DENEY}")
print(f"DOĞRULUK: {correct}/{total} = {100*correct/total:.1f}%")
print("="*60)